In [2]:

import os
import requests
from getpass import getpass
from google import genai
from langchain_core.tools import Tool


if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API Key: ")

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter Tavily API Key: ")


client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"]
)

MODEL = "gemini-3.5-flash"


def run_qa(query):
    response = client.models.generate_content(
        model=MODEL,
        contents=f"Answer clearly: {query}"
    )
    return response.text or ""

qa_tool = Tool(
    name="Simple QA",
    func=run_qa,
    description="Answer factual questions clearly"
)


def run_summary(text):
    response = client.models.generate_content(
        model=MODEL,
        contents=f"Summarize the following text:\n\n{text}"
    )
    return response.text or ""

summary_tool = Tool(
    name="Summarizer",
    func=run_summary,
    description="Summarizes input text"
)


def run_web_search(query):
    url = "https://api.tavily.com/search"

    payload = {
        "api_key": os.environ["TAVILY_API_KEY"],
        "query": query,
        "max_results": 3
    }

    response = requests.post(
        url,
        json=payload,
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    results = data.get("results", [])

    if not results:
        return "No search results found."

    formatted_results = []

    for result in results:
        formatted_results.append(
            f"Title: {result.get('title', '')}\n"
            f"URL: {result.get('url', '')}\n"
            f"Content: {result.get('content', '')}"
        )

    return "\n\n".join(formatted_results)

search_tool = Tool(
    name="Web Search",
    func=run_web_search,
    description="Search the internet for current information using Tavily"
)

# =====================================
# 6. Test All Three Tools
# =====================================

qa_query = "What is LangGraph in LangChain?"

summary_text = """
LangGraph is a framework for building stateful multi-step agents using LangChain.
It uses graph-based design to model agent workflows and memory.
"""

search_query = "Latest updates on GPT-4o by OpenAI"

print("\nSimple QA Tool Output:\n")
print(qa_tool.invoke(qa_query))

print("\nSummarizer Tool Output:\n")
print(summary_tool.invoke(summary_text))

print("\nWeb Search Tool Output:\n")
print(search_tool.invoke(search_query))


Enter Tavily API Key: ··········

Simple QA Tool Output:

**LangGraph** is a framework built by the LangChain team designed to create **complex, stateful, multi-agent AI applications using graph-based logic.**

While standard LangChain is great for linear, step-by-step sequences (e.g., Step A $\rightarrow$ Step B $\rightarrow$ Step C), **LangGraph allows for loops, cycles, and persistence.** 

---

### Key Features of LangGraph

1. **Cycles and Loops (Agentic Behavior):** 
   Real AI agents need to iterate—try an action, check the output, fix errors, and try again. Standard chains cannot easily loop, but LangGraph allows workflows to loop back to previous steps until a task is successfully completed.

2. **Built-in State Management:** 
   LangGraph maintains a centralized "State" object (like a shared memory) that tracks the conversation history, variables, and context across all nodes in the graph.

3. **Human-in-the-Loop:** 
   It natively supports pausing a workflow to wait for huma